# BerryDB SDK walkthrough: one profile per student

This notebook runs the same flow as the [berry-sdk-sample-app](https://github.com/sushilkrs/berry-sdk-sample-app)
web app, step by step, with the BerryDB Python SDK (`BerryDbNew`):

1. set up two BerryDB databases: one record **per document**, one record **per student**;
2. register students with a permanent **UUID generated by your app**;
3. send a resume PDF to BerryDB, which extracts **one structured record** from it;
4. merge each student's documents into **one profile**, saved with `upsert` (id = student UUID);
5. run **named-entity recognition** (NER) on a document;
6. see what happens when **two students upload the same file**.

**You need:** a BerryDB API key (account settings at [app.berrydb.io](https://app.berrydb.io)).
Run the cells in order. The notebook creates its own sample PDFs, so no files are needed.

> Tip: in Colab, store the key as a secret named `BERRYDB_API_KEY` (key icon in the left sidebar);
> otherwise the notebook asks for it.

In [ ]:
!pip install -q BerryDbNew==1.7.7 requests

## Settings and connection

The SDK host is `https://api.berrydb.io` (`app.berrydb.io` is the web app, not the API).
The database and schema names match the sample web app, so you can open the app afterwards and keep going.

In [ ]:
import hashlib, json, os, time, uuid
from getpass import getpass

import requests
from berrydb import BerryDB

BERRYDB_HOST = "https://api.berrydb.io"
DOCS_DB, DOCS_SCHEMA = "ResumePlaygroundKG", "ResumeSchema"              # one record per uploaded PDF
PROFILES_DB, PROFILES_SCHEMA = "StudentProfiles", "StudentProfileSchema"  # one record per student
REPO_RAW = "https://raw.githubusercontent.com/sushilkrs/berry-sdk-sample-app/main"


def _api_key():
    if os.environ.get("BERRYDB_API_KEY"):
        return os.environ["BERRYDB_API_KEY"]
    try:
        from google.colab import userdata  # Colab secret, if you added one
        return userdata.get("BERRYDB_API_KEY")
    except Exception:
        return getpass("BerryDB API key: ")


API_KEY = _api_key()
BerryDB.init(BERRYDB_HOST)
print("Connected. Databases visible to this key:", len(BerryDB.databases(API_KEY) or {}))

## Step 1. Create the schemas and databases (once per organization)

A **schema** is the form BerryDB fills in: the fields a record has and their types.
A **database** is a named collection of records that follow one schema.

| Database | Schema | One record per | Record id |
|---|---|---|---|
| `ResumePlaygroundKG` | `ResumeSchema` | uploaded PDF | chosen by BerryDB from the file's bytes |
| `StudentProfiles` | `StudentProfileSchema` | student | the student's UUID, chosen by your app |

The schema definitions come from the sample app's repository. Anything that already exists is skipped.

In [ ]:
def schema_exists(name):
    r = requests.get(f"{BERRYDB_HOST}/profile/schema", params={"schemaName": name, "apiKey": API_KEY}, timeout=30)
    return r.status_code == 200 and "schema" in (r.json() or {})


def create_schema(name, schema):
    """The SDK's create_schema first; if the server rejects its body, the flat body it accepts."""
    if BerryDB.create_schema(API_KEY, name, schema["description"], schema):
        return "created with the SDK"
    body = {"name": name, "description": schema["description"], "rawSchema": schema["rawSchema"],
            "dataTypes": schema["schemaDataTypes"], "filterFields": schema["filterFields"],
            "indexFields": schema["indexFields"], "displayTextField": schema["displayTextField"],
            "displayImageField": schema["displayImageField"], "embeddedType": False}
    r = requests.post(f"{BERRYDB_HOST}/profile/schema", params={"apiKey": API_KEY}, json=body, timeout=60)
    r.raise_for_status()
    return "created with the REST API (the SDK call was rejected)"


for schema_name, db_name in [(DOCS_SCHEMA, DOCS_DB), (PROFILES_SCHEMA, PROFILES_DB)]:
    if schema_exists(schema_name):
        print(f"schema {schema_name}: already exists")
    else:
        definition = requests.get(f"{REPO_RAW}/backend/berry_setup/{schema_name}.json", timeout=30).json()
        print(f"schema {schema_name}:", create_schema(schema_name, definition))
    if db_name in (BerryDB.databases(API_KEY) or {}):
        print(f"database {db_name}: already exists")
    else:
        BerryDB.create_database(API_KEY, db_name, schema_name)
        print(f"database {db_name}: created")

docs_db = BerryDB.connect(API_KEY, DOCS_DB)
profiles_db = BerryDB.connect(API_KEY, PROFILES_DB)

## Step 2. Register students

Your app, not BerryDB, decides who a student is. Each student gets a permanent **UUID** the moment they
register. Never identify a student by a file name, file content or the name written in a document:
two students can upload the very same file.

Saving the profile uses `upsert`: if no record has that id, BerryDB **creates** it; if one exists, BerryDB
**replaces** it (it does not merge, so always send the full record).

In [ ]:
students = {
    "A": {"id": str(uuid.uuid4()), "full_name": "Ava Patel", "email": "ava.patel@example.com"},
    "B": {"id": str(uuid.uuid4()), "full_name": "Ben Carter", "email": ""},
}
documents = {"A": [], "B": []}   # what your app keeps: one entry per upload
created_doc_records = set()      # for the clean-up step at the end


def save_profile(profile):
    profiles_db.upsert([profile])   # id = student UUID: created the first time, replaced afterwards


for key, s in students.items():
    save_profile({"id": s["id"], "studentId": s["id"], "fullName": s["full_name"], "email": s["email"],
                  "skillsList": [], "workExperience": [], "education": [], "documentIds": []})
    print(f"Student {key}: {s['full_name']}  UUID {s['id']}  -> profile created in {PROFILES_DB}")

## Step 3. Make sample resume PDFs

To keep the notebook self-contained, this cell writes two small text PDFs for Ava: an older resume and an
updated one. (To use your own file instead, read its bytes, e.g. `open("cv.pdf", "rb").read()`, or upload it
in Colab with `google.colab.files.upload()`.)

In [ ]:
def make_pdf(lines):
    """A minimal one-page PDF with a real text layer (BerryDB refuses scans without text)."""
    esc = lambda s: s.replace("\\", "\\\\").replace("(", "\\(").replace(")", "\\)")
    body = "BT /F1 11 Tf 50 790 Td 14 TL " + " ".join(f"({esc(l)}) Tj T*" for l in lines) + " ET"
    objs = ["<< /Type /Catalog /Pages 2 0 R >>", "<< /Type /Pages /Kids [3 0 R] /Count 1 >>",
            "<< /Type /Page /Parent 2 0 R /MediaBox [0 0 612 842] /Contents 4 0 R /Resources << /Font << /F1 5 0 R >> >> >>",
            f"<< /Length {len(body)} >>\nstream\n{body}\nendstream", "<< /Type /Font /Subtype /Type1 /BaseFont /Helvetica >>"]
    data, offsets = b"%PDF-1.4\n", []
    for i, o in enumerate(objs, 1):
        offsets.append(len(data))
        data += f"{i} 0 obj\n{o}\nendobj\n".encode("latin-1")
    xref = len(data)
    data += f"xref\n0 {len(objs) + 1}\n0000000000 65535 f \n".encode() + "".join(f"{o:010d} 00000 n \n" for o in offsets).encode()
    return data + f"trailer\n<< /Size {len(objs) + 1} /Root 1 0 R >>\nstartxref\n{xref}\n%%EOF\n".encode()


resume_old = make_pdf([
    "Ava Patel", "ava.patel@example.com | Ahmedabad, India | github.com/ava-patel-dev",
    "Software Engineer",
    "Summary: Backend developer building REST services in Python and PostgreSQL.",
    "Experience",
    "Cloudnest (2021 - 2024), Software Engineer: built Django REST APIs; added caching with Redis.",
    "Education", "B.E. Information Technology, Gujarat Technological University, 2021",
    "Skills: Python, Django, PostgreSQL, Redis, Docker",
])
resume_new = make_pdf([
    "Ava Patel - Resume (updated)", "ava.patel@example.com | Pune, India",
    "Senior Software Engineer",
    "Summary: Senior engineer designing event-driven services in Python and Go on AWS.",
    "Experience",
    "Streamline Labs (2024 - present), Senior Software Engineer: moved services to Go and Kafka on AWS.",
    "Cloudnest (2021 - 2024), Software Engineer: Django REST APIs and Redis caching.",
    "Education", "M.S. Computer Science, Georgia Tech (online), 2024",
    "Skills: Go, Python, Kafka, AWS, Kubernetes, PostgreSQL",
])
print(f"old resume: {len(resume_old)} bytes, new resume: {len(resume_new)} bytes")

## Step 4. Send a PDF to BerryDB for structured extraction

`POST /berrydb/ingest/structured` reads the PDF, fills in the fields of the database's schema with AI and
writes **one record**. The `metadata` you send is stored on that record: here, whose document it is
(`studentId`) and which upload it was (`documentId`).

> The SDK's `db.ingest_structured(urls)` does the same for PDFs at a public or presigned URL. To send the
> file itself, call the endpoint directly as below.

In [ ]:
def ingest(pdf_bytes, file_name, student_key):
    student_id, document_id = students[student_key]["id"], str(uuid.uuid4())
    r = requests.post(
        f"{BERRYDB_HOST}/berrydb/ingest/structured",
        headers={"x-api-key": API_KEY},
        data={"database_name": DOCS_DB,
              "metadata": json.dumps({"studentId": student_id, "documentId": document_id})},
        files={"file": (file_name, pdf_bytes, "application/pdf")},
        timeout=300,
    )
    result = r.json()
    per_file = (result.get("per_file") or [{}])[0]
    if r.status_code != 200 or per_file.get("error"):
        raise RuntimeError(per_file.get("error") or result)
    documents[student_key].append({"documentId": document_id, "file_name": file_name,
                                   "sha256": hashlib.sha256(pdf_bytes).hexdigest(), "berry_record_id": per_file["id"]})
    created_doc_records.add(per_file["id"])
    return per_file


res = ingest(resume_old, "resume.pdf", "A")
print("BerryDB record id:", res["id"])
print("fields filled:", res["fields_filled"], "| pages:", res["pages"], "| full text stored in:", res["text_field"])

Read the record back with the SDK. BerryDB reads are **eventually consistent**: a record written a moment
ago can take a few seconds to appear, so read by id and retry briefly.

In [ ]:
def get_record(db, record_id, attempts=6):
    for i in range(attempts):
        docs = db.get_all_documents(document_ids=[record_id]) or []
        if docs:
            return docs[0]
        time.sleep(2 * (i + 1))
    raise RuntimeError(f"{record_id} not readable yet")


rec = get_record(docs_db, res["id"])
for field in ["fullName", "jobTitle", "location", "email", "skillsList", "degreeName", "institutionName",
              "studentId", "documentId"]:
    print(f"{field:16} {rec.get(field)}")
print("work experience:", [w.get("company") for w in rec.get("workExperience") or []])

## Step 5. Merge the student's documents into one profile

BerryDB makes one record per **file** and never combines them, so your app does. These are the sample app's
rules: registration name and email win; for single values the newest upload wins; lists are combined without
duplicates; one work entry per company. The profile is rebuilt from all documents and saved whole.

In [ ]:
SINGLE = ["jobTitle", "location", "professionalSummary", "linkedInUrl", "githubUrl", "portfolioUrl"]
norm = lambda s: " ".join(str(s or "").lower().split())


def build_profile(student_key):
    s = students[student_key]
    p = {"id": s["id"], "studentId": s["id"], "fullName": s["full_name"], "email": s["email"],
         **{f: "" for f in SINGLE}, "skillsList": [], "workExperience": [], "education": [], "documentIds": []}
    for doc in documents[student_key]:                       # oldest first, so newer values overwrite
        rec = get_record(docs_db, doc["berry_record_id"])
        p["documentIds"].append(doc["documentId"])
        p.update({f: rec[f] for f in SINGLE if rec.get(f)})
        p["email"] = p["email"] or rec.get("email") or ""
        for skill in rec.get("skillsList") or []:
            if norm(skill) not in map(norm, p["skillsList"]):
                p["skillsList"].append(skill)
        for w in rec.get("workExperience") or []:
            entry = {"company": w.get("company") or "", "details": w.get("details") or "", "documentId": doc["documentId"]}
            p["workExperience"] = [e for e in p["workExperience"] if norm(e["company"]) != norm(entry["company"])] + [entry]
        if rec.get("degreeName") or rec.get("institutionName"):
            p["education"].append({"degreeName": rec.get("degreeName") or "", "institutionName": rec.get("institutionName") or "",
                                   "graduationYear": rec.get("graduationYear") or 0, "documentId": doc["documentId"]})
    p["updatedAt"] = time.strftime("%Y-%m-%dT%H:%M:%SZ", time.gmtime())
    return p


def show_profile(student_key):
    p = get_record(profiles_db, students[student_key]["id"])
    print(f"profile id {p['id']}  (= student UUID)")
    print("headline:", p.get("jobTitle"), "·", p.get("location"))
    print("skills:  ", p.get("skillsList"))
    print("work:    ", [w["company"] for w in p.get("workExperience") or []])
    print("degrees: ", [e["degreeName"] for e in p.get("education") or []])
    print("built from", len(p.get("documentIds") or []), "document(s)")


save_profile(build_profile("A"))
time.sleep(3)
show_profile("A")

## Step 6. The same student uploads another document

Ava uploads her updated resume. BerryDB extracts it into a **new** document record; your app rebuilds her
profile from both documents and upserts it with the **same id**, so BerryDB replaces the existing profile.

In [ ]:
res2 = ingest(resume_new, "resume_updated.pdf", "A")
print("new document record:", res2["id"])
save_profile(build_profile("A"))
time.sleep(3)
show_profile("A")   # same profile id, now combining both resumes

## Step 7. Find entities (NER)

NER marks the names of things in text: PERSON, ORGANIZATION, LOCATION, DATE… With `annotate=True`, BerryDB
saves the results as **annotations** on the record, in the background. It runs on a field the schema declares;
`content.text` holds the document's full text.

In [ ]:
record_id = documents["A"][-1]["berry_record_id"]
print("NER job:", docs_db.ner("content.text", document_ids=[record_id], annotate=True))

entities = []
for _ in range(10):
    time.sleep(3)
    rec = get_record(docs_db, record_id)
    entities = [e for a in rec.get("annotations") or [] if a.get("task") == "ner"
                for r in a.get("results") or [] for e in r.get("output") or []]
    if entities:
        break
unique = {(e["start"], e["end"], e["label"]): e for e in entities}   # BerryDB may repeat a span
for e in sorted(unique.values(), key=lambda e: int(e["start"])):
    print(f"{e['label']:13} {e['text']:28} characters {e['start']}-{e['end']}")

## Step 8. Two students upload the same file

Ben uploads a file byte-for-byte identical to Ava's first resume. Compare files only **within one student**
(your app's SHA-256 check), never across students, so this upload is allowed.

BerryDB derives a document record's id from the database and the file's bytes, so Ben's upload lands on the
**same** document record as Ava's: it is re-labelled with Ben's `studentId` and its annotations are replaced.
Each student's **profile** is still built separately and stays correct.

In [ ]:
res_b = ingest(resume_old, "my_resume.pdf", "B")
ava_first = documents["A"][0]["berry_record_id"]
print("Ava's first document record:", ava_first)
print("Ben's document record:      ", res_b["id"], "-> same record:", res_b["id"] == ava_first)
shared = get_record(docs_db, res_b["id"])
print("record now tagged studentId =", "Ben" if shared.get("studentId") == students["B"]["id"] else "Ava")

save_profile(build_profile("B"))
time.sleep(3)
print("\nBen's profile:"); show_profile("B")
print("\nAva's profile is unchanged:"); show_profile("A")

## Step 9. Clean up (optional)

Deletes the records this notebook created: both students' profiles and their document records.
The databases and schemas stay, ready for the web app or another run.

In [ ]:
for record_id in created_doc_records:
    print("delete document record", record_id, docs_db.delete_document(record_id))
for s in students.values():
    print("delete profile", s["id"], profiles_db.delete_document(s["id"]))

## Optional: run the full web app inside Colab

Starts the sample app (FastAPI backend + web UI) on this Colab machine and opens it in a new browser tab.
It uses the same databases, so you can add students, upload your own PDFs and run NER from the UI.
Files and the app's SQLite database live on this Colab machine and disappear when the session ends.

In [ ]:
import subprocess, sys

!git clone -q https://github.com/sushilkrs/berry-sdk-sample-app.git /content/berry-sdk-sample-app
!pip install -q -r /content/berry-sdk-sample-app/backend/requirements.txt
with open("/content/berry-sdk-sample-app/.env", "w") as f:
    f.write(f"BERRYDB_HOST={BERRYDB_HOST}\nBERRYDB_API_KEY={API_KEY}\n")

server = subprocess.Popen([sys.executable, "-m", "backend"], cwd="/content/berry-sdk-sample-app",
                          stdout=open("/content/app.log", "w"), stderr=subprocess.STDOUT)
for _ in range(30):
    try:
        if requests.get("http://127.0.0.1:8800/api/health", timeout=2).ok:
            break
    except requests.RequestException:
        time.sleep(1)

from google.colab import output
output.serve_kernel_port_as_window(8800)   # click the link to open the app; stop it with server.terminate()